<a href="https://colab.research.google.com/github/genaivicky/Master-Llamaindex/blob/main/docs/examples/agent/agents_as_tools.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multi-Agent Report Generation using Agents as Tools

<a href="https://colab.research.google.com/github/run-llama/llama_index/blob/main/docs/examples/agent/agents_as_tools.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In this notebook, we will explore how to create a multi-agent system that uses a top-level agent to orchestrate a group of agents as tools. Specifically, we will create a system that can research, write, and review a report on a given topic.

This notebook will assume that you have already either read the [basic agent workflow notebook](https://docs.llamaindex.ai/en/stable/examples/agent/agent_workflow_basic) or the [general agent documentation](https://docs.llamaindex.ai/en/stable/understanding/agent/).

## Setup

In this example, we will use `OpenAI` as our LLM. For all LLMs, check out the [examples documentation](https://docs.llamaindex.ai/en/stable/examples/llm/openai/) or [LlamaHub](https://llamahub.ai/?tab=llms) for a list of all supported LLMs and how to install/use them.

If we wanted, each agent could have a different LLM, but for this example, we will use the same LLM for all agents.

In [2]:
%pip install llama-index

In [1]:
!pip install "jedi>=0.16"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 26.4 MB/s eta 0:00:00


In [69]:
import os
import getpass

print("Please enter your OpenAI API Key:")
os.environ["OPENAI_API_KEY"] = getpass.getpass()

print("\nPlease enter your Tavily API Key:")
os.environ["TAVILY_API_KEY"] = getpass.getpass()

print("\nBoth keys saved to environment securely!")

Please enter your OpenAI API Key:
··········

Please enter your Tavily API Key:
··········

Both keys saved to environment securely!


In [71]:
from llama_index.llms.openai import OpenAI

sub_agent_llm = OpenAI(model="gpt-5-nano", api_key=os.environ["OPENAI_API_KEY"])
orchestrator_llm = OpenAI(model="gpt-5.4", api_key=os.environ["OPENAI_API_KEY"])

## System Design

Our system will have three agents:

1. A `ResearchAgent` that will search the web for information on the given topic.
2. A `WriteAgent` that will write the report using the information found by the `ResearchAgent`.
3. A `ReviewAgent` that will review the report and provide feedback.

We will then use a top-level agent to orchestrate the other agents to write our report.

While there are many ways to implement this system, in this case, we will use a single `web_search` tool to search the web for information on the given topic.


In [57]:
%pip install tavily-python

In [72]:
from tavily import AsyncTavilyClient


async def search_web(query: str) -> str:
    """Useful for using the web to answer questions."""
    client = AsyncTavilyClient(api_key=os.environ["TAVILY_API_KEY"])
    return str(await client.search(query))

With our tool defined, we can now create our sub-agents.

If the LLM you are using supports tool calling, you can use the `FunctionAgent` class. Otherwise, you can use the `ReActAgent` class.

In [73]:
from llama_index.core.agent.workflow import FunctionAgent, ReActAgent

research_agent = FunctionAgent(
    system_prompt=(
        "You are the ResearchAgent that can search the web for information on a given topic and record notes on the topic. "
        "You should output notes on the topic in a structured format."
    ),
    llm=sub_agent_llm,
    tools=[search_web],
)

write_agent = FunctionAgent(
    system_prompt=(
        "You are the WriteAgent that can write a report on a given topic. "
        "Your report should be in a markdown format. The content should be grounded in the research notes. "
        "Return your markdown report surrounded by <report>...</report> tags."
    ),
    llm=sub_agent_llm,
    tools=[],
)

review_agent = FunctionAgent(
    system_prompt=(
        "You are the ReviewAgent that can review the write report and provide feedback. "
        "Your review should either approve the current report or request changes to be implemented."
    ),
    llm=sub_agent_llm,
    tools=[],
)

With our sub-agents defined, we can then convert them into tools that can be used by the top-level agent.

In [74]:
import re
from llama_index.core.workflow import Context


async def call_research_agent(ctx: Context, prompt: str) -> str:
    """Useful for recording research notes based on a specific prompt."""
    result = await research_agent.run(
        user_msg=f"Write some notes about the following: {prompt}"
    )

    async with ctx.store.edit_state() as ctx_state:
        ctx_state["state"]["research_notes"].append(str(result))

    return str(result)


async def call_write_agent(ctx: Context) -> str:
    """Useful for writing a report based on the research notes or revising the report based on feedback."""
    async with ctx.store.edit_state() as ctx_state:
        notes = ctx_state["state"].get("research_notes", None)
        if not notes:
            return "No research notes to write from."

        user_msg = f"Write a markdown report from the following notes. Be sure to output the report in the following format: <report>...</report>:\n\n"

        # Add the feedback to the user message if it exists
        feedback = ctx_state["state"].get("review", None)
        if feedback:
            user_msg += f"<feedback>{feedback}</feedback>\n\n"

        # Add the research notes to the user message
        notes = "\n\n".join(notes)
        user_msg += f"<research_notes>{notes}</research_notes>\n\n"

        # Run the write agent
        result = await write_agent.run(user_msg=user_msg)

        # --- THE FIX: Safe Regex Extraction ---
        match = re.search(r"<report>(.*?)</report>", str(result), re.DOTALL)

        # 🔥 ADD THIS LINE: Force print the raw output immediately!
        print("\n\n🔥 [DEBUG] Write Agent successfully generated text! Preview:\n", str(result)[:500], "...\n")

        if match:
            report_text = match.group(1)
        else:
            # Fallback: Use the whole response if the tags are missing
            report_text = str(result)

        ctx_state["state"]["report_content"] = report_text

    return report_text


async def call_review_agent(ctx: Context) -> str:
    """Useful for reviewing the report and providing feedback."""
    async with ctx.store.edit_state() as ctx_state:
        report = ctx_state["state"].get("report_content", None)
        if not report:
            return "No report content to review."

        result = await review_agent.run(
            user_msg=f"Review the following report: {report}"
        )
        ctx_state["state"]["review"] = result

    return result

## Creating the Top-Level Orchestrator Agent

With our sub-agents defined as tools, we can now create our top-level orchestrator agent.

In [75]:
orchestrator = FunctionAgent(
    name="orchestrator", # Make sure it has a name
    system_prompt=(
        "You are an expert Orchestrator in charge of report writing. "
        "You MUST follow this exact sequence strictly: "
        "1. You MUST call the `call_research_agent` tool first. "
        "2. You MUST call the `call_write_agent` tool second. "
        "3. You MUST call the `call_review_agent` tool last. "
        "CRITICAL: Do NOT notify the user that the report is ready until you have actually executed these tools and received their results. Do not make up or fake the completion."
    ),
    llm=orchestrator_llm,
    tools=[
        call_research_agent,
        call_write_agent,
        call_review_agent,
    ],
    initial_state={
        "research_notes": [],
        "report_content": None,
        "review": None,
    },
)

## Running the Agent

Let's run our agents! We can iterate over events as the workflow runs.

In [76]:
from llama_index.core.agent.workflow import (
    AgentInput,
    AgentOutput,
    ToolCall,
    ToolCallResult,
    AgentStream,
)
from llama_index.core.workflow import Context

# Create a context for the orchestrator to hold history/state
ctx = Context(orchestrator)


# 1. Update your run function to use the orchestrator directly and forcefully initialize the state
async def run_orchestrator(user_msg: str):
    # Initialize Context directly on the orchestrator
    ctx = Context(orchestrator)

    # Failsafe: Force the state to exist so our tools NEVER throw a KeyError
    await ctx.store.set("state", {
        "research_notes": [],
        "report_content": None,
        "review": None,
    })

    # Run the orchestrator directly
    handler = orchestrator.run(
        user_msg=user_msg,
        ctx=ctx,
    )

    async for event in handler.stream_events():
        if isinstance(event, AgentStream):
            if event.delta:
                print(event.delta, end="", flush=True)
        elif isinstance(event, AgentOutput):
            if event.tool_calls:
                print("\n🛠️  Planning to use tools:", [call.tool_name for call in event.tool_calls])
        elif isinstance(event, ToolCallResult):
            print(f"\n🔧 Tool Result ({event.tool_name}): Successfully completed.")
        elif isinstance(event, ToolCall):
            print(f"\n🔨 Calling Tool: {event.tool_name}")

    # Await the execution loop to finalize state
    await handler
    return ctx

# 2. Execution Cell
prompt = (
    "Write me a report on the history of the internet. "
    "Briefly describe the history of the internet, including the development of the internet, the development of the web, "
    "and the development of the internet in the 21st century."
)

print("Starting Orchestrator...\n")
# Run and capture the context
final_ctx = await run_orchestrator(prompt)

# 3. Extract the final report safely
state_data = await final_ctx.store.get("state")

print("\n\n" + "="*50)
print("📝 EXTRACTED FINAL REPORT:")
print("="*50)
if state_data and state_data.get("report_content"):
    print(state_data["report_content"])
else:
    print("❌ Critical Failure: The report_content is still empty.")

Starting Orchestrator...


🛠️  Planning to use tools: ['call_research_agent']

🔨 Calling Tool: call_research_agent

🔧 Tool Result (call_research_agent): Successfully completed.

🛠️  Planning to use tools: ['call_write_agent']

🔨 Calling Tool: call_write_agent


🔥 [DEBUG] Write Agent successfully generated text! Preview:
 <report>
# The Evolution of the Internet and the World Wide Web: A Concise Synthesis

This report synthesizes the development of the Internet from its origins in packet-switching networks to the global Web, and its continued evolution into the 21st century. It draws on foundational milestones, core technologies, and governance and policy considerations outlined in the supplied notes.

## 1) Development of the Internet (origin to commercialization)

- Origins and core principle
  - ARPANET and ea ...


🔧 Tool Result (call_write_agent): Successfully completed.

🛠️  Planning to use tools: ['call_review_agent']

🔨 Calling Tool: call_review_agent

🔧 Tool Result (call_review

In [53]:
await run_orchestrator(
    ctx=ctx,
    user_msg=(
        "Write me a report on the history of the internet. "
        "Briefly describe the history of the internet, including the development of the internet, the development of the web, "
        "and the development of the internet in the 21st century."
    ),
)

With our report written and revised/reviewed, we can inspect the final report in the state.

In [78]:
print("\n\n" + "="*50)
print("📝 EXTRACTED FINAL REPORT:")
print("="*50)

# The foolproof way to access the state dictionary
async with final_ctx.store.edit_state() as ctx_state:
    # Safely pull the state dictionary, defaulting to an empty dict if missing
    state_data = ctx_state.get("state", {})
    report = state_data.get("report_content")

    if report:
        print(report)
    else:
        print("❌ Critical Failure: The report_content is empty.")



📝 EXTRACTED FINAL REPORT:

# The Evolution of the Internet and the World Wide Web: A Concise Synthesis

This report synthesizes the development of the Internet from its origins in packet-switching networks to the global Web, and its continued evolution into the 21st century. It draws on foundational milestones, core technologies, and governance and policy considerations outlined in the supplied notes.

## 1) Development of the Internet (origin to commercialization)

- Origins and core principle
  - ARPANET and early concept (late 1960s): Funded by U.S. DARPA/ARPA to explore packet-switching as a robust method for interconnecting computers across campuses; 1969 marks the first ARPANET node connections.
  - Packet switching as a core principle: Breaks data into small packets for efficient, resilient transmission; foundational to later network design.

- Early protocols, experiments, and growth
  - Early protocols and host-to-host experiments (1970s): Email and other experiments demonst